# INTRODUCTION

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
This Machine Learning project leverages a comprehensive US Real Estate dataset scraped from Realtor.com, containing over 2.2 million listing records across key spatial, temporal, and physical dimensions. Originating as an Exploratory Data Analysis (EDA) of real estate dynamics, the study has expanded into an end-to-end predictive modeling framework designed to estimate real estate listing prices across diverse US markets. By combining rigorous data preprocessing, feature engineering, spatial aggregation, and advanced regression algorithms, this project addresses significant dataset challenges—such as extreme skewness, missing values, and high-cardinality spatial identifiers—to deliver accurate, scalable, and interpretable price predictions.

</div>

# OBJECTIVES:

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">

<ul>
<li>Data Preprocessing: Clean the dataset by handling missing values, removing duplicates, filtering extreme outliers, and reformatting date/categorical fields.

<li>Univariate Analysis: Evaluate the overall distribution, skewness, and central tendencies of key numerical features such as price, house_size, and acre_lot.

<li>Geographical Insights: Analyze price variations and price-per-square-foot trends across different states, cities, and zip codes to identify high-value versus affordable markets.

<li>Correlation & Relationship Mapping: Determine the strength of relationships between physical property features (bedrooms, bathrooms, living area) and final listing prices.

<li>Temporal Trend Analysis: Explore historical market behavior using prev_sold_date to observe sales activity patterns over time.

<li>Feature Engineering: Create derived metrics like price_per_sqft and total_rooms to prepare a clean, structured baseline for future machine learning models.

<li> Train and evaluate supervised linear regression baseline models on the engineered features to accurately predict property listing prices and assess performance using metrices like RMSE and MAE.
</div>

<div
>

# 1. Collection & Inspection


## 1.1 Loading Libraries and Dataset

In [1]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, FunctionTransformer
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = pd.read_csv('/home/kartika/Datasets/realtor-usarealestatedata.csv')
data.to_parquet(
    '/home/kartika/Datasets/realtor-usarealestatedata.parquet',
    engine='pyarrow',
    index=False
)

for col in data.select_dtypes(include = ['object', 'str']).columns :
    data[col] = data[col].astype('category')

for col in data.select_dtypes(include = ['int', 'float']).columns :
    data[col] = pd.to_numeric(data[col], downcast = 'integer' if 'int' in str(data[col].dtype) else 'float')



print(f"Memory Usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [ ]:
def out_per(col = None, df = None):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    per = len(df[(df[col] < lower_bound) | (df[col] > upper_bound)]) / len(df) * 100
    high = len(df[df[col] > upper_bound])/ len(df) * 100
    low = len(df[df[col] < lower_bound])/ len(df) * 100
    print(f"Percentage of outliers in {col}: {per:.2f}%")
    print(f"Percentage of Extreme High Outliers in {col}: {high:.2f}%")
    print(f"Percentage of Extreme Low Outliers in {col}: {low:.2f}%")

<div>

<div>

## 1.2 Basic Inspection

In [ ]:
data.head(5)


In [ ]:
data.shape

In [ ]:
data.info()


In [ ]:
data.isnull().sum()

In [ ]:
(data.isnull().sum()/len(data))*100

In [ ]:
data.duplicated().sum()

In [ ]:
data['bed'] = pd.to_numeric(data['bed'], errors = 'coerce').astype('float64')
data['bath'] = pd.to_numeric(data['bath'], errors = 'coerce').astype('float64')

<div>

# 2. Cleaning & Wrangling

## 2.1 Dropping Irrelevant Fields

In [ ]:
data.drop(columns = ['brokered_by', 'street'], inplace = True)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify"> 'brokered_by' and 'street' are removed as irrelevant field. 

In [ ]:
data.dropna(subset = ['city', 'state', 'zip_code'], inplace = True)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">< 0.01% values are missing, so dropped them.

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Dropping irrelevant rows or fields and duplicates are done before splitting data.

In [ ]:
data['prev_sold_date'] = pd.to_datetime(data['prev_sold_date'], errors = 'coerce')
## errors = coerce, valid date --> converted to datetime,
## null/Nan --> remains Nan


<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Also converted prev_sold_date to datetime, preserving the null values.

In [ ]:
mask = ~((data['acre_lot'] > 10000) & (data['price'] < 100000))
data = data[mask].copy()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Dropped rows where price < $10000(standard minimum house price).

<div>

## 2.2 Train-Test-Validate Split

In [ ]:
train, temp = train_test_split(data, test_size = 0.4, random_state = 42)
validate, test = train_test_split(temp, test_size = 0.5, random_state = 42)


print(f"Train: {len(train)}, {len(train)/len(data)*100}%\n Validate: {len(validate)},{len(validate)/len(data)*100}% \n Test: {len(test)}, {len(test)/len(data)*100}%")

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Splitted original data into train, test, validate (60/20/20) prior to data pre-processing and feature engineering to prevent data leakage.

<div>

## 2.3 Handaling Missing Values

In [ ]:
for df in [train, test, validate]:
    df = df.dropna(subset = ['price'], inplace = True)



In [ ]:
train.drop(train['price'].idxmax(), inplace = True)
## Other dependent features do not justify the price

In [ ]:
#train['price'].isnull().sum()
#validate['price'].isnull().sum()
#test['price'].isnull().sum()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify"><1% values are missing in price field. Since price is the target field, so they are dropped instead of imputation. 


<div>

In [ ]:
city_acre_lot_median = train.groupby('city', observed = True)['acre_lot'].median()
state_acre_lot_median = train.groupby('state', observed = True)['acre_lot'].median()
global_acre_lot_median = train['acre_lot'].median()

for df in [train, test, validate]:
    df['acre_lot'] = df['acre_lot'].fillna(df['city'].map(city_acre_lot_median))
    df['acre_lot'] = df['acre_lot'].fillna(df['state'].map(state_acre_lot_median))
    df['acre_lot'] = df['acre_lot'].fillna(global_acre_lot_median)

In [ ]:
#train['acre_lot'].isnull().sum()
#validate['acre_lot'].isnull().sum()
#test['acre_lot'].isnull().sum()

In [ ]:
train.drop(train['acre_lot'].idxmax(), inplace = True)
## Invalid as per location and corresponding price




<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Around 15% values are missing in acre_lot, so imputed with group median.

<div>

In [ ]:
bins = list(range(0, 10000, 500)) + [np.inf]
train['sqft_bin'] = pd.cut(train['house_size'], bins=bins)
test['sqft_bin'] = pd.cut(test['house_size'], bins=bins)
validate['sqft_bin'] = pd.cut(validate['house_size'], bins=bins)


sqft_bed_bin_median = train.groupby('sqft_bin', observed = True)['bed'].median()
sqft_bath_bin_median = train.groupby('sqft_bin', observed = True)['bath'].median()
global_bed_median = train['bed'].median()
global_bath_median = train['bath'].median()

for df in [train, test, validate]:
    df['bed'] = df['bed'].fillna(df['sqft_bin'].map(sqft_bed_bin_median))
    df['bed'] = df['bed'].fillna(global_bed_median)


for df in [train, test, validate]:
    df['bath'] = df['bath'].fillna(df['sqft_bin'].map(sqft_bath_bin_median))
    df['bath'] = df['bath'].fillna(global_bath_median)


In [ ]:
#train['bed'].isnull().sum()
#validate['bath'].isnull().sum()
#test['bed'].isnull().sum()

In [ ]:
train.drop('sqft_bin', axis = 1, inplace = True)

In [ ]:
validate.drop('sqft_bin', axis = 1, inplace = True)

In [ ]:
test.drop('sqft_bin', axis = 1, inplace = True)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">~22% values are missing in bed and ~23% values are missing in bath, so imputed with house_size(sqft_bins). Dropped sqft_bin after imputation.
<br>Dropped invalid rows, where house size to bed ratio was <120, which is non standard for typical houses or apartments.
<br> DRopped invalid rows, where house size to bath ratio was < 40, which is non standard for typical houses or apartments.
Bed and bath flag functions are called are after handaling missing values to prevent dropping of unnecessary rows.

<div>

In [ ]:
house_size_group_median = train.groupby(
    ["zip_code", "bed", "bath"], observed=True
)["house_size"].median()
global_house_size_median = train["house_size"].median()

keys = ["zip_code", "bed", "bath"]
for df in [train, test, validate]:
    key_index = pd.MultiIndex.from_frame(df[keys])
    mapped = key_index.map(house_size_group_median)
    df["house_size"] = df["house_size"].fillna(
        pd.Series(mapped, index=df.index)
    )
    df["house_size"] = df["house_size"].fillna(global_house_size_median)


In [ ]:
#train['house_size'].isnull().sum()
#validate.house_size.isnull().sum()
#test['house_size'].isnull().sum()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Around 25% values are missing in house_size, so imputed with group median from 'zip_code', 'bed', and 'bath'.


## 2.4 Flag Functions, Cleaning (After Handaling Missing Values)

In [ ]:
def flag_bed(df, sqft_col = 'house_size', bed_col = 'bath', min_ratio = 40):
    ratio = df[sqft_col] /df[bed_col]
    keep = ratio >=min_ratio

    dropped_ids = df.index[~keep]
    print(f"Dropped {len(dropped_ids)} rows: {list(dropped_ids)}")
    return df[keep].copy()


train = flag_bed(train)
test = flag_bed(test)
validate = flag_bed(validate)

In [ ]:
def flag_bed(df, sqft_col = 'house_size', bed_col = 'bed', min_ratio = 120):
    ratio = df[sqft_col] /df[bed_col]
    keep = ratio >=min_ratio

    dropped_ids = df.index[~keep]
    print(f"Dropped {len(dropped_ids)} rows: {list(dropped_ids)}")
    return df[keep].copy()


train = flag_bed(train)
test = flag_bed(test)
validate = flag_bed(validate)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Dropped rows where house_size to bed ratio <120 and house_size to bath ratio < 40 as invalid entries.

In [ ]:
def house_plot_flag(df, house_col = 'house_size', plot_col = 'acre_lot'):

    is_valid = df[house_col] <= (df[plot_col] * 43560)
    dropped_ids = df[~is_valid].index.tolist()

    print(f"Dropped {len(dropped_ids)} rows. Dropped rows: {dropped_ids}")

    return df[is_valid].copy()

train = house_plot_flag(train)
test = house_plot_flag(test)
validate = house_plot_flag(validate)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Dropped rows where house size(sqft) > land size(acre * 45560)sqft.

<div>

# 3. Exploratory Data Analysis

# 3.1 Univarite Analysis



## 3.1.1 Numerical Fields

In [ ]:
train['price'].describe()

In [ ]:
train['price'].skew()

In [ ]:
out_per('price', train)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16,6))

sns.histplot(
    x = train['price'],
    bins = 120,
    kde = True,
    color = 'green',
    ax = ax[0],

)
ax[0].set_title('Shape of Price Distribution')

sns.boxplot(
    data = train,
    y = 'price',
    color = 'blue',
    showmeans = True,
    ax = ax[1]
)
ax[1].set_title('Price Distribution')


plt.suptitle('Price Distribution')
sns.despine()
plt.show()

In [ ]:
np.log1p(train['price']).skew()

In [ ]:
sns.histplot(
    np.log1p(train['price']),
    kde = True
)

plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Most of the houses have mid-range to budget price. The distribution is extremely positively skewed, with most the houses having price around half million, while few houses priced extremely high(billions). The outliers are extremely high outliers, as the price of house in top 25% jumps to million to billion.
<br>Log transformation supressed extremely high values and hence the data is now slightly negatively skewed.

<div>

In [ ]:
train['bed'].describe() 

In [ ]:
train['bath'].describe()

In [ ]:
out_per('bath', train)

In [ ]:
out_per('bed', train)

In [ ]:
train['bath'].skew()

In [ ]:
train['bed'].skew()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16,6))

sns.boxplot(
    y = train['bed'],
    showmeans = True,
    color = 'green',
    ax = ax[0],

)
ax[0].set_title('Bedroom Distribution')

sns.boxplot(
    y = train['bath'],
    showmeans = True,
    color = 'blue',
    ax = ax[1]
)
ax[1].set_title('Bathrooms Distribution')


plt.suptitle('Beds and Baths Distribution')
sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
The bathroom distribution among house property is extremely positively skewed, majority of housese have 1-2 bedrooms, while rest have large number of bathrooms.
<br>
The bedroom distribution among house property is extremely positively skewed, majority of housese have 1-4 bedrooms, while rest have extremely large number of bedrooms. The bedroom skewness is different from bathrooms, as some houses have low numbers of bedrooms compared to central value. Hence, in bedroom there is presence of both extreme high and low outliers.

In [ ]:
train['bed'] = pd.to_numeric(train['bed'], errors = 'coerce').astype('float64')
train['bath'] = pd.to_numeric(train['bath'], errors = 'coerce').astype('float64')

In [ ]:

print(np.log1p(train['bed']).skew())
print(np.log1p(train['bath']).skew())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16,6))

sns.histplot(
    np.log1p(train['bed']),
    kde = True,
    ax = ax[0],

)
ax[0].set_title('Log Transformation on Bedroom')


sns.histplot(
    np.log1p(train['bath']),
    kde = True,
    ax = ax[1]
)
ax[1].set_title('Log Transformation on Bathroom')


sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Performed log transformation on bed, the distribution is now slightly negatively skewed as it contains extreme low outliers as well.
Transformation on bath, the distribution is slighty positively skewed.

<div>

In [ ]:
train['acre_lot'].describe()

In [ ]:
train['house_size'].describe()

In [ ]:
train['acre_lot'].skew()

In [ ]:
train['house_size'].skew()

In [ ]:
out_per('acre_lot', train)

In [ ]:
out_per('house_size', train)

In [ ]:
fig, ax = plt.subplots(1,2, figsize = (20,8))

sns.boxplot(
    data = train,
    y = 'acre_lot',
    color = 'blue',
    showmeans= True,
    ax = ax[0]
)

sns.boxplot(
    data = train, 
    y = 'house_size',
    color = 'cyan',
    showmeans = True,
    ax = ax[1]
    
)

plt.suptitle('Land Size(Acres) and House Size(sqft) Distribution')
plt.show()



<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
The land size distribution is extremely positivley skewed. Majority of land size is less than or equal to 0.75 acres, while rest properties have extremely large land size.
<br>The house size is also extremely positively skewed. The house size distribution contains both extremely high and low outliers, will be handled during data preprocessing.

In [ ]:
print(np.log1p(train['acre_lot']).skew())
print(np.log1p(train['house_size']).skew())

In [ ]:
fig, ax = plt.subplots(1,2, figsize = (20,8))

sns.histplot(
    np.log1p(train['acre_lot']),
    kde = True,
    color = 'green',
    ax = ax[0]
)

sns.histplot(
    np.log1p(train['house_size']),
    kde = True,
    color = 'cyan',
    ax = ax[1]
    
)

plt.suptitle('Land Size(Acres) and House Size(sqft) Distribution After Log Transformation')
plt.show()



<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
After log transformation, acre lot is still heavily positively skewed, while house size is slightly positively skewed.

<divv>

## 3.1.2 Categorical Fields

In [ ]:
train['status'].value_counts()

In [ ]:
plt.pie(
    train['status'].value_counts().values,
    labels = train['status'].value_counts().index,
    autopct = '%1.2f%%',
    shadow = True,
    startangle = 90,
    colors = plt.cm.Greens(np.linspace(1, 0.3, 3))
)

plt.title('Property Status')
sns.despine()
plt.show()


<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">62.38% of the property have already built house, which are ready for sale. 36.5% of the property houses have been already sold and rest less than 2% of the property represent vacant land ready for construction.

<div>

In [ ]:
train['city'] = train['city'].cat.remove_unused_categories()
train['city'].value_counts()

In [ ]:
top_cities = train['city'].value_counts().head(10)
top_cities_sum = top_cities.sum()
top_cities_pct = (top_cities_sum/len(train))*100
top_cities_pct

In [ ]:
train['state'] = train['state'].cat.remove_unused_categories()
train['state'].value_counts()

In [ ]:
len(train['state'].value_counts())

In [ ]:
top_states = train['state'].value_counts().head(10)
top_states_sum = top_states.sum()
top_states_pct = (top_states_sum/len(train))*100
top_states_pct

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 6))

ax[0].barh(
    y = train['city'].value_counts().head(10).index,
    width = train['city'].value_counts().head(10).values,
    color = plt.cm.viridis(np.linspace(1, 0.9, 10 )),
    linewidth = 1.5,
    edgecolor = 'cyan'
)
ax[0].set_title('Top Cities By Number of Properties')
ax[0].set_ylabel('Number of Properties')
ax[0].set_xlabel('Cities')
ax[0].invert_yaxis()



ax[1].barh(
    y = train['state'].value_counts().head(10).index,
    width = train['state'].value_counts().head(10).values,
    color = plt.cm.viridis(np.linspace(1, 0.9, 10 )),
    linewidth = 1.5,
    edgecolor = 'cyan'
)
ax[1].set_title('Top States By Number of Properties')
ax[1].set_ylabel('Number of Properties')
ax[1].set_xlabel('States')
ax[1].invert_yaxis()



plt.subplots_adjust(wspace = 0.9)
sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
<ul>
<li>Houston, Chicago and New York City are top three cities by number of property. Florida, California and Texas are top three states by number of properties.
<li>Also, around 56% of the properties come from top 10 states, while only 5.6% of the properties come from top 10 cities.
<li>This indicates city-level distribution is highly spread out, while state-level distribution is much more concentrated.
<li>State is a much more dominant geographic feature than city.
</ul>
</div>

<div>

## 3.1.3 Date Time Fields

In [ ]:
train['prev_sold_date'].isnull().sum()/len(train) * 100

In [ ]:
train[(train['prev_sold_date'].isnull()) & (train['prev_sold_date'] == 'sold')]

In [ ]:
plt.pie(
    [32.89, 67.11],
    labels = ['Prev Not Sold', 'Prev Sold'],
    colors = ['pink', 'lightgreen'],
    shadow = True,
)

plt.title('Is Prev Sold ?')
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Is prev sold date is missing where the property has not been sold previously, so all missing values are intentional. This field will be used during feature engineering to create is_prev_sold field.

<div>

# 3.2 Bivariate Analysis

## 3.2.1 Target vs Numerical

In [ ]:
train['bed'].corr(train['price'])

In [ ]:
train['bath'].corr(train['price'])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16,6))


# Bedroom vs Price
sns.scatterplot(
    data = train,
    x = 'bed',
    y = 'price',
    color = 'green',
    linewidth = 1.5,
    edgecolor = 'blue',
    ax = ax[0],
    alpha = 0.5
)
ax[0].set_title('Bedroom vs Price')

# Bathroom vs Price
sns.scatterplot(
    data = train,
    x = 'bath',
    y = 'price',
    linewidth = 1.5,
    edgecolor = 'blue',
    color = 'green',
    ax = ax[1],
    alpha = 0.5,
)
ax[1].set_title('Bathroom vs Price')


plt.suptitle('House Profile vs Price')
plt.subplots_adjust(wspace = 0.9)
sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify"> There is a weak-to-moderate positive relationship between the number of bathrooms and price, bedrooms account for only 9% of the variation in house price.
<br>
There is extremely weak(negligible) relationship between number of bathrooms and price, bathrooms account for only ~2.56% of the variation in house price.

<div>

In [ ]:
corr1 = train['price'].corr(train['acre_lot'])

In [ ]:
print(corr1)

In [ ]:
print((corr1**2)*100)

In [ ]:
corr2 = train['price'].corr(train['house_size'])
corr2

In [ ]:
print((corr2**2)*100)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16,6))


# Land Size vs Price
sns.scatterplot(
    data = train,
    x = 'acre_lot',
    y = 'price',
    color = 'green',
    linewidth = 1.5,
    edgecolor = 'blue',
    ax = ax[0],
    alpha = 0.5
)
ax[0].set_title('Land Size (acre) vs Price')

# House Size vs Price
sns.scatterplot(
    data = train,
    x = 'house_size',
    y = 'price',
    linewidth = 1.5,
    edgecolor = 'blue',
    color = 'green',
    ax = ax[1],
    alpha = 0.5
)
ax[1].set_title('House Size vs Price')


plt.suptitle('Hosue Size vs Price')
plt.subplots_adjust(wspace = 0.9)
sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
There is extremely weak correlation between house size, acrelot and price.

<div>

## 3.2.2 Categorical vs Target (Price)

In [ ]:
state_price = train.groupby('state', observed = True)['price'].agg(['count', 'mean', 'median']).sort_values('median', ascending = False)
top_states = state_price.head(10).reset_index()
top_states

In [ ]:
city_price = train.groupby('city', observed = True)['price'].agg(['count', 'mean', 'median']).sort_values('median', ascending = False)
top_city = city_price.head(10).reset_index()
top_city

In [ ]:
fig, ax = plt.subplots(1,2, figsize = (12, 6))

ax[0].barh(
    y = top_states['state'],
    width = top_states['median'],
    color = plt.cm.magma(np.linspace(1, 0.9, 10 )),
    linewidth = 1.5,
    edgecolor = 'cyan'
)
ax[0].set_title('Top States By Median Price')
ax[0].set_ylabel('States')
ax[0].set_xlabel('Median Price ($)')
ax[0].invert_yaxis()


ax[1].barh(
    y = top_city['city'],
    width = top_city['median'],
    color = plt.cm.magma(np.linspace(1, 0.9, 10 )),
    linewidth = 1.5,
    edgecolor = 'cyan'
)
ax[1].set_title('Top Cities By Median Price')
ax[1].set_ylabel('Cities')
ax[1].set_xlabel('Median Price ($)')
ax[1].invert_yaxis()

sns.despine()
plt.subplots_adjust(wspace = 0.9)
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
There is greater price variance at city level compared to state level.<br>
City is much higher-cardinality and higher-variance feature than state.

In [ ]:
status_price = train.groupby('status', observed = True)['price'].agg(['count', 'mean', 'median']).sort_values('median', ascending = False)
status_price = status_price.reset_index()
status_price

In [ ]:
sns.barplot(
    data = status_price,
    x = 'status',
    y = 'median',
    hue = 'count',
    palette = 'viridis'
)

plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Ready to build properties have comparatively high median price.

<div>

## 3.2.3 Date Time vs Target(Price)

In [ ]:
train['sold_year'] = train['prev_sold_date'].dt.year
train['sold_month'] = train['prev_sold_date'].dt.month
train['sold_quarter'] = train['prev_sold_date'].dt.quarter
train['has_prev_sold'] = train['prev_sold_date'].notna()

In [ ]:
year_price = train.groupby('sold_year')['price'].agg(['count', 'median', 'mean']).reset_index()
year_price

In [ ]:
month_price = train.groupby('sold_month')['price'].agg(['count', 'median', 'mean']).reset_index()
month_price

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
fig, ax = plt.subplots(2, 1, figsize=(12, 8))


year_price['sold_year'] = pd.to_numeric(year_price['sold_year'], errors='coerce')
year_price = year_price.dropna(subset=['sold_year'])
year_price = year_price[(year_price['sold_year'] >= 1910) & (year_price['sold_year'] <= 2026)]
year_price = year_price.sort_values('sold_year')


sns.lineplot(
    data=year_price,
    x='sold_year',
    y='median',
    marker='o',
    ax=ax[0]
)
ax[0].set_title('Median Price by Previous Sale Year')
ax[0].set_xlabel('Previous Sale Year')
ax[0].set_ylabel('Median Price ($)')
ax[0].set_xticks(ticks=range(1925, 2027, 10))

sns.lineplot(
    data=month_price,
    x='sold_month',
    y='median',
    marker='o',
    ax=ax[1]
)
ax[1].set_title('Median Price by Previous Sale Month')
ax[1].set_xlabel('Previous Sale Month')
ax[1].set_ylabel('Median Price ($)')  
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax[1].set_xticks(range(1, 13), labels=months)

plt.subplots_adjust(hspace=0.4)  
plt.tight_layout()
sns.despine()
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Median property prices show extreme hike at 1928, then stable from 1935-1950, high-low from 1950-1970 and then follow stable/consistent pricing with bit rise from 2021.
<br>Median price hikes in May and July and gradually drops till November.
<br>This indicates price is influenced by date(year, month, quarter).

In [ ]:
quarter_price = train.groupby('sold_quarter')['price'].agg(['count', 'mean', 'median']).reset_index()
quarter_price

In [ ]:
sns.set_theme(style = 'whitegrid', palette = 'muted')
fig, ax1 = plt.subplots(figsize = (10, 8))


sns.lineplot(
    data = quarter_price,
    x = 'sold_quarter',
    y = 'median',
    marker = 'o',
    color = 'green',
    ax = ax1,
    label = 'Median Price',

)
ax1.set_xlabel('Previous Sale Quarter')
ax1.set_ylabel('Median Price($)')
ax1.xaxis.set_major_locator(ticker.MultipleLocator(1))

ax2 = ax1.twinx()

sns.lineplot(
    data = quarter_price,
    x = 'sold_quarter',
    y = 'count',
    marker = 'o',
    ax = ax2,
    color = 'orange',
    label = 'Count of Properties Sold'
)
ax2.set_ylabel('Count of Properties Sold')
ax2.grid(False)

plt.title('Median Price and Count of Properties Sold by Previous Sale Quarter')
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Most properties have been sold in the first quarter, while the median price of properties sold in third quarter is highest.

In [ ]:
prev_sold = train.groupby('has_prev_sold', observed = True)['price'].agg(['count', 'mean', 'median']).reset_index()
prev_sold

In [ ]:
sns.barplot(
    data = prev_sold,
    x = 'has_prev_sold',
    y = 'median',
    hue = 'count',
    palette = 'viridis'
)

plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Around 67% of the properties have been previously sold properties and have comparatively high price.

In [ ]:
train.drop(columns = ['has_prev_sold', 'sold_year', 'sold_month', 'sold_quarter'], inplace = True)

<div>

# 3.3 Multivariate/ Multicollinearity

## 3.3.1 Numerical Fields

In [ ]:
num_cols = train.select_dtypes(include = ['int', 'float', 'object']).columns

corr_matrix = train[num_cols].corr() * 100
corr_matrix

In [ ]:
plt.figure(figsize = (8,6))

sns.heatmap(corr_matrix, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -100, vmax = 100)
plt.title('Numerical Feature Correlation Heatmap')
plt.show()

In [ ]:
sns.pairplot(train[num_cols])

plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Bath and bath are strongly related to each other. They can be combined into single feature (like total_rooms or bath_to_bed_ratio).

<div>

# 5. Data Pre-Processing & Feature Engineering

In [ ]:
feat_train = train.copy() 

## 5.1 Outliers Handaling

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Since the numerical features are mostly skewed, they are detected and handled by IQR method. Alter, log1p transform and then z-score test method.

In [ ]:
out_per('price', feat_train)

In [ ]:
out_per('bed', feat_train)

In [ ]:
out_per('bath', feat_train)

In [ ]:
out_per('house_size', feat_train)

In [ ]:
out_per('acre_lot', feat_train)

In [ ]:
def out_capper(col = None, df = None, train_df = None): 
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)

    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return df[col].clip(lower = lower_bound, upper = upper_bound)

    # return np.where(df[col] > upper_bound, upper_bound, np.where(df[col] < lower_bound, lower_bound, df[col]))



for df in [feat_train, validate, test]:
    for col in ['bed', 'bath', 'house_size', 'acre_lot']:
        df[col] = out_capper(col, df, feat_train)


In [ ]:
print(f"Percentage of outliers in bed after capping: {feat_train['bed'].skew()}% {'\n'} Percentage of outliers in bath after capping: {feat_train['bath'].skew()}% {'\n'} Percentage of outliers in house_size after capping: {feat_train['house_size'].skew()}% {'\n'} Percentage of outliers in acre_lot after capping: {feat_train['acre_lot'].skew()}%")

## 5.2 Log Transformation

In [ ]:
log_features = ['price', 'acre_lot', 'house_size']

for df in [feat_train, validate, test]:
    for col in log_features:
        df[col] = np.log1p(df[col])

<div>

## 5.3 New Features

<div>

In [ ]:
co_matrix = (
    train.assign(total_rooms = train['bed'] + train['bath'],
                 bed_to_bath = train['bed'] /train['bath'],
                 )
    .select_dtypes(include = ['number'])
    .corr() 
)

sns.heatmap(co_matrix, annot = True, fmt = '.2f', cmap = 'coolwarm', vmin = -1, vmax = 1)
plt.title('Numerical Feature Correlation Heatmap')
plt.figure(figsize = (12, 8))
plt.show()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">bath has strongest correlation with price among bed, bed to bath ratio and total rooms.

In [ ]:
feat_train['prev_sold_year'] = feat_train['prev_sold_date'].dt.year         
feat_train['prev_sold_month'] = feat_train['prev_sold_date'].dt.month
feat_train['prev_sold_quarter'] = feat_train['prev_sold_date'].dt.quarter
def date_features(df = None, col = 'prev_sold_date'):
    return df.assign(
        prev_sold_year = df[col].dt.year,
        prev_sold_month = df[col].dt.month,
        prev_sold_quarter = df[col].dt.quarter
    )

test = date_features(test)
validate = date_features(validate)

In [ ]:
features = ['prev_sold_year', 'prev_sold_month', 'prev_sold_quarter']

imputer = SimpleImputer(strategy = 'median')

feat_train[features] = imputer.fit_transform(feat_train[features])
test[features] = imputer.transform(test[features])
validate[features] = imputer.transform(validate[features])

In [ ]:
feat_train['has_prev_sold'] = feat_train['prev_sold_date'].notna().astype(int)

validate['has_prev_sold'] = validate['prev_sold_date'].notna().astype(int)
test['has_prev_sold'] = test['prev_sold_date'].notna().astype(int)

In [ ]:
feat_train.drop(columns = ['prev_sold_date'], inplace = True)
test.drop(columns = ['prev_sold_date'], inplace = True)
validate.drop(columns = ['prev_sold_date'], inplace = True)

In [ ]:
for df in [feat_train, validate, test]:    
    df['month_sin'] = np.sin(2 * np.pi * df['prev_sold_month'] / 12)   
    df['month_cos'] = np.cos(2 * np.pi * df['prev_sold_month'] / 12)

In [ ]:
feat_train.drop(columns = ['prev_sold_month'], inplace = True)
test.drop(columns = ['prev_sold_month'], inplace = True)
validate.drop(columns = ['prev_sold_month'], inplace = True)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">price_per_sqft is more correlated with price, than price_per_acre.

<div>

## 5.4 Categorical Encoding

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Since all the categorical fields in this dataset are nominal categories, one-hot encoding and target encoding is done.


In [ ]:
feat_train['zip_code'] = feat_train['zip_code'].astype('category')
test['zip_code'] = test['zip_code'].astype('category')
validate['zip_code'] = validate['zip_code'].astype('category')

In [ ]:
transformer = ColumnTransformer(
    transformers = [
        ('city_encoded', TargetEncoder(smoothing = 30), ['city']),
        ('state_encoded', TargetEncoder(smoothing = 4), ['state']),
        ('zipcode_encoded', TargetEncoder(smoothing = 5), ['zip_code']),
        ('status_encoded', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'), ['status'])],
        remainder = 'passthrough')

In [ ]:
log_transformer = FunctionTransformer(func = np.log1p, inverse_func = np.expm1)


In [ ]:
x_train = feat_train.drop(columns = ['price'])
y_train = feat_train['price']

x_test = test.drop(columns = ['price'])
y_test = test['price']


x_validate = validate.drop(columns = ['price'])
y_validate = validate['price']


In [ ]:
x_train_encoded = transformer.fit_transform(x_train, y_train)
x_test_encoded = transformer.transform(x_test)
x_validate_encoded = transformer.transform(x_validate)

In [ ]:
feature_names = transformer.get_feature_names_out()

x_train_encoded_df = pd.DataFrame(x_train_encoded, columns = feature_names, index = x_train.index)
x_test_encoded_df = pd.DataFrame(x_test_encoded, columns = feature_names, index = x_test.index)
x_validate_encoded_df = pd.DataFrame(x_validate_encoded, columns = feature_names, index = x_validate.index)


## 5.5 Feature Scaling(Standardization)

In [ ]:
scaler = StandardScaler()

In [ ]:
x_train_scaled = scaler.fit_transform(x_train_encoded_df)
x_test_scaled = scaler.transform(x_test_encoded_df)
x_validate_scaled = scaler.transform(x_validate_encoded_df)


In [ ]:
cols = scaler.get_feature_names_out()


In [ ]:
x_train_scaled_df = pd.DataFrame(x_train_scaled, columns = cols, index = x_train.index)
x_test_scaled_df = pd.DataFrame(x_test_scaled, columns = cols, index = x_test.index)
x_validate_scaled_df = pd.DataFrame(x_validate_scaled, columns = cols, index = x_validate.index)

In [ ]:
x_train_scaled_df

<div>

# 6. Model Training and Evaluation

## 6.1 Linear Regression

In [ ]:
reg_model = LinearRegression()

In [ ]:
reg_model.fit(x_train_scaled_df, y_train)

In [ ]:
y_train_log_pred = reg_model.predict(x_train_scaled_df)
y_validate_log_pred = reg_model.predict(x_validate_scaled_df)

In [ ]:
y_train_pred = np.expm1(y_train_log_pred)
y_validate_pred = np.expm1(y_validate_log_pred)

y_train_exp = np.expm1(y_train)
y_validate_exp = np.expm1(y_validate)

In [ ]:

def evaluate(y_true, y_pred):
    print('MAE: ', mean_absolute_error(y_true, y_pred))
    print('RMSE: ', np.sqrt(mean_squared_error(y_true, y_pred)))
    print('r2_score: ', r2_score(y_true, y_pred))

In [ ]:
evaluate(y_train, y_train_log_pred)

In [ ]:
print('Training Performance')
evaluate(y_train_exp, y_train_pred)

print('\n')
print('Validating Performance')
evaluate(y_validate_exp, y_validate_pred)

## 6.2 Visualizing Model Performance

In [ ]:
plt.figure(figsize = (8,6))

sns.scatterplot(
    x = y_validate_exp,
    y = y_validate_pred,
    alpha = 0.3
)

plt.xlabel("Actual Price")
plt.ylabel("Predicated Price")

sns.despine()
plt.show()

In [ ]:
residuals = y_validate_exp - y_validate_pred

sns.scatterplot(
    x = y_validate_pred,
    y = residuals,
    alpha = 0.3
)


plt.axhline(0, color = 'red', linestyle = '--')

plt.xlabel('Predicated Price')
plt.ylabel('Residue')
plt.show()

In [ ]:
coefficients = pd.DataFrame({
    'Feature' : x_train_scaled_df.columns,
    'coeff' : reg_model.coef_
})

coefficients['abs_coeff'] = (coefficients['coeff'].abs())

coefficients.sort_values('abs_coeff', ascending = False)